<div style="background:linear-gradient(135deg,#001F3F 0%,#0093D5 100%);padding:40px 32px;border-left:6px solid #EE3A43;">
<h1 style="color:#fff;font-family:Arial,sans-serif;font-size:28px;margin:0 0 8px;">07 — Anomaly Detection</h1>
<h2 style="color:rgba(255,255,255,.75);font-family:Arial,sans-serif;font-size:16px;font-weight:400;margin:0 0 20px;">SpiriCom · NOC Intelligence Platform · Huawei Technologies Tunisia</h2>
<div style="display:flex;gap:32px;flex-wrap:wrap;">
<div style="color:rgba(255,255,255,.65);font-size:13px;"><strong style="color:#fff;">Methods</strong><br/>Z-Score · IQR · Isolation Forest · Consensus</div>
<div style="color:rgba(255,255,255,.65);font-size:13px;"><strong style="color:#fff;">Sources</strong><br/>complaints · KPIs · churn features · scores</div>
<div style="color:rgba(255,255,255,.65);font-size:13px;"><strong style="color:#fff;">Output</strong><br/>models/anomaly/anomaly_results.parquet</div>
</div></div>

---
## What is Anomaly Detection?

Anomaly detection finds **data points that deviate significantly from expected behavior**.
In telecom NOC context, an anomaly is any network event, customer pattern, or complaint
surge that falls outside the normal operating range and may require operator attention.

### Use Cases in SpiriCam

| # | Use Case | Data Source | Signal |
|---|----------|-------------|--------|
| UC-1 | Complaint spike | complaints_clean.parquet | Daily count > 2σ |
| UC-2 | Network KPI degradation | kpi_clean.parquet | Latency / packet loss spike |
| UC-3 | Traffic anomaly | churn_features.parquet | Abnormal 5G/4G volume |
| UC-4 | Churn risk surge | churn_scores.parquet | Risk level distribution shift |
| UC-5 | Geographic hotspot | complaints + region | One province suddenly dominant |
| UC-6 | Brand anomaly | churn_features by brand | One brand's churn rate jumps |

### Methods Used

| Method | Type | Best for | SpiriCam use |
|--------|------|----------|--------------|
| Z-Score | Statistical | Normally distributed features | KPI metrics (latency, loss) |
| IQR | Statistical | Skewed distributions | Traffic volumes, complaint counts |
| Isolation Forest | ML (ensemble) | Multivariate, high-dimensional | Full feature matrix anomalies |
| Consensus | Combined | Reliable production flags | Final `anomaly_flag` for dashboard |

> **Why Consensus?** Any single method has false positives.
> Flagging only events where *both* statistical AND Isolation Forest agree
> gives high-precision alerts suitable for a NOC dashboard.

In [1]:
# ══════════════════════════════════════════════════════════════════════
# 0. IMPORTS, PATHS & STYLE
# ══════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy  as np
import json, warnings, joblib
from pathlib  import Path
from datetime import datetime, timedelta

from sklearn.ensemble        import IsolationForest
from sklearn.preprocessing   import StandardScaler

import matplotlib.pyplot  as plt
import matplotlib.patches as mpatches
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Paths ──────────────────────────────────────────────────────────────
DATA_DIR  = Path('data')
PROC_DIR  = Path('data/processed')
OUT_DIR   = Path('data/outputs')
MODEL_DIR = Path('models')
ANOM_DIR  = MODEL_DIR / 'anomaly'
FIG_DIR   = OUT_DIR  / 'figures'
for d in [ANOM_DIR, FIG_DIR]: d.mkdir(parents=True, exist_ok=True)

# ── Huawei Palette ────────────────────────────────────────────────────
HW = dict(
    blue='#0093D5', red='#EE3A43', navy='#001F3F', cyan='#00C3FF',
    green='#22C55E', amber='#F59E0B', purple='#8B5CF6', muted='#6B7280',
)
plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.edgecolor':'#E5E7EB', 'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'axes.grid.axis':'y', 'grid.color':'#F3F4F6',
    'axes.labelcolor':HW['navy'], 'axes.labelweight':'bold',
    'axes.titlesize':13, 'axes.titleweight':'bold', 'axes.titlecolor':HW['navy'],
    'xtick.color':HW['muted'], 'ytick.color':HW['muted'],
    'figure.dpi':110, 'savefig.dpi':300, 'savefig.bbox':'tight',
})

def save_fig(name):
    path = FIG_DIR / f'{name}.png'
    plt.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')

def watermark(fig):
    fig.text(0.99, 0.01, 'SpiriCom · Huawei Technologies Tunisia',
             ha='right', va='bottom', fontsize=7, color=HW['muted'], style='italic')

print('✅ Setup complete')

✅ Setup complete


---
## Section 1 — Load Data Sources

In [2]:
# ── 1.1  Load all available data sources ──────────────────────────────
datasets = {}

# Complaints
for p in [OUT_DIR/'complaints_clean.parquet', DATA_DIR/'complaints_clean.parquet']:
    if p.exists():
        datasets['complaints'] = pd.read_parquet(p)
        datasets['complaints'].columns = datasets['complaints'].columns.str.lower()
        print(f'  complaints   : {len(datasets["complaints"]):,} rows  cols={list(datasets["complaints"].columns[:6])}...')
        break

# NB07-FIX-1: updated to v6 pipeline outputs.
# churn_labelled_v6.parquet covers all 4,896 subscribers with de-imputed
# QoS columns (e2e_delay_ms, client_rtt_ms, etc.) — preferred for anomaly
# detection since it spans the full population, not just labelled customers.
# churn_features_v6.parquet is tried as a fallback (2,566 labelled only).
for p in [PROC_DIR/'churn_labelled_v6.parquet',
          PROC_DIR/'churn_features_v6.parquet',
          OUT_DIR/'churn_features.parquet',     # legacy fallback
          PROC_DIR/'churn_features.parquet']:
    if p.exists():
        datasets['kpi'] = pd.read_parquet(p)
        datasets['kpi'].columns = datasets['kpi'].columns.str.lower()
        print(f'  kpi          : {len(datasets["kpi"]):,} rows  ({p.name})')
        print(f'  kpi cols     : {list(datasets["kpi"].columns[:8])}...')
        break

# NB07-FIX-1b: risk scores — use calibrated v6 output.
# v6 schema: msisdn | risk (float) | risk_band (low/medium/high) | split | churn
# Legacy schema: msisdn | churn_probability | risk_level (CRITICAL/HIGH/..)
for p in [OUT_DIR/'risk_scores_v2.parquet',
          MODEL_DIR/'disengagement_risk_scores_v2.parquet',
          MODEL_DIR/'disengagement_risk_scores.parquet',
          OUT_DIR/'churn_scores.parquet',        # legacy fallback
          MODEL_DIR/'churn_scores.parquet']:
    if p.exists():
        datasets['scores'] = pd.read_parquet(p)
        datasets['scores'].columns = datasets['scores'].columns.str.lower()
        print(f'  risk scores  : {len(datasets["scores"]):,} rows  ({p.name})')
        print(f'  score cols   : {list(datasets["scores"].columns)}')
        break

print(f'\nLoaded {len(datasets)} dataset(s): {list(datasets.keys())}')

  kpi          : 4,896 rows  (churn_labelled_v6.parquet)
  kpi cols     : ['msisdn', 'session_flag', 'traffic_5g', 'dou_total', 'duration', 'brand', 'dou_total_imputed', 'duration_imputed']...
  risk scores  : 2,566 rows  (risk_scores_v2.parquet)
  score cols   : ['msisdn', 'risk', 'split', 'churn', 'risk_band']

Loaded 2 dataset(s): ['kpi', 'scores']


In [3]:
# ── 1.2  Print FULL column list for each dataset ─────────────────────
# (The display above truncates at col 6 — this shows everything)
for name, df in datasets.items():
    print(f'\n{name.upper()} columns ({len(df.columns)}):')  
    for i, col in enumerate(df.columns):
        print(f'  {i:2d}  {col}')
    print(f'  dtype summary: {df.dtypes.value_counts().to_dict()}')



KPI columns (34):
   0  msisdn
   1  session_flag
   2  traffic_5g
   3  dou_total
   4  duration
   5  brand
   6  dou_total_imputed
   7  duration_imputed
   8  traffic_5g_imputed
   9  traffic_4g_imputed
  10  traffic_3g_imputed
  11  traffic_2g_imputed
  12  e2e_delay_ms_imputed
  13  client_rtt_ms_imputed
  14  c1_low_usage
  15  c2_low_dur
  16  churn
  17  generation
  18  layer2name
  19  mobility_class
  20  usertype
  21  user_class
  22  tertype
  23  e2e_delay_ms
  24  client_rtt_ms
  25  server_rtt_ms
  26  client_packet_loss_rate
  27  server_packet_loss_rate
  28  page_response_delay
  29  page_download_throughput
  30  dns_delay
  31  dns_sr
  32  tcp_connection_sr
  33  number_of_regions
  dtype summary: {dtype('float64'): 15, dtype('int64'): 11, dtype('O'): 8}

SCORES columns (5):
   0  msisdn
   1  risk
   2  split
   3  churn
   4  risk_band
  dtype summary: {dtype('O'): 2, dtype('float64'): 1, dtype('int64'): 1, CategoricalDtype(categories=['low', 'medium', 'high'

---
## Section 2 — Statistical Methods: Z-Score and IQR

### How Z-Score works
```
z = (x − μ) / σ

If |z| > threshold (typically 2.5–3.0) → anomaly
```
A Z-score measures how many **standard deviations** a value is from the mean.
Best for normally distributed metrics like latency and packet loss.

### How IQR works
```
IQR = Q3 − Q1
Lower fence = Q1 − 1.5 × IQR
Upper fence = Q3 + 1.5 × IQR

Values outside fences → anomaly
```
IQR is **resistant to extreme outliers** because it uses percentiles, not the mean.
Best for skewed distributions like traffic volumes and complaint counts.

In [4]:
# ── 2.1  Statistical anomaly detection functions ───────────────────────

def zscore_anomaly(series: pd.Series, threshold: float = 2.5) -> pd.Series:
    """
    Return boolean Series: True where |z-score| > threshold.
    Robust to NaN — fills with False.
    """
    mu  = series.mean()
    std = series.std()
    if std == 0:
        return pd.Series(False, index=series.index)
    z = (series - mu) / std
    return z.abs() > threshold


def iqr_anomaly(series: pd.Series, multiplier: float = 1.5) -> pd.Series:
    """
    Return boolean Series: True where value is outside
    [Q1 - multiplier*IQR, Q3 + multiplier*IQR].
    Use multiplier=3.0 for a more conservative (fewer) flags.
    """
    q1  = series.quantile(0.25)
    q3  = series.quantile(0.75)
    iqr = q3 - q1
    lo  = q1 - multiplier * iqr
    hi  = q3 + multiplier * iqr
    return (series < lo) | (series > hi)


def stat_anomaly_score(df: pd.DataFrame, features: list[str],
                       threshold_z: float = 2.5,
                       iqr_mult: float = 1.5) -> pd.Series:
    """
    Combined statistical anomaly:
      - Count how many features are anomalous per row
      - Row flagged if ANY feature is an anomaly
    Returns: (anomaly_flag Series, driver Series)
    """
    flag_matrix = pd.DataFrame(index=df.index)
    for feat in features:
        if feat not in df.columns:
            continue
        col = df[feat].fillna(df[feat].median())
        z_flag   = zscore_anomaly(col, threshold_z)
        iqr_flag = iqr_anomaly(col, iqr_mult)
        # Use OR: either method flagging is enough for a feature to be 'anomalous'
        flag_matrix[feat] = z_flag | iqr_flag

    anomaly_flag = flag_matrix.any(axis=1).astype(int)
    # Top driver = feature with most anomalies that is True for this row
    driver = flag_matrix.idxmax(axis=1).where(anomaly_flag == 1, other='')
    score  = flag_matrix.sum(axis=1) / max(len(features), 1)  # 0–1 normalised
    return anomaly_flag, driver, score


print('✅ Statistical functions defined')
print('   zscore_anomaly(series, threshold=2.5)')
print('   iqr_anomaly(series, multiplier=1.5)')
print('   stat_anomaly_score(df, features, threshold_z, iqr_mult)')

✅ Statistical functions defined
   zscore_anomaly(series, threshold=2.5)
   iqr_anomaly(series, multiplier=1.5)
   stat_anomaly_score(df, features, threshold_z, iqr_mult)


In [5]:
# ── 2.2  Apply statistical detection to KPI features ──────────────────
# Telecom KPI data is highly skewed — use conservative thresholds:
#   threshold_z = 3.5  (standard uses 2.5, but that flags 64% here)
#   iqr_mult    = 3.0  (standard uses 1.5, but generates too many false positives)
# Target: stat flags ~10-15% so consensus with IF is meaningful.

if 'kpi' in datasets:
    kpi = datasets['kpi'].copy()

    # NB07-FIX-2: STAT_FEATURES aligned to v6 column names
    # (churn_labelled_v6.parquet QoS columns). Legacy names kept as
    # fallback so the notebook still runs against pre-v6 datasets.
    STAT_FEATURES = [
        f for f in [
            # v6 QoS names (churn_labelled_v6 / churn_features_v6)
            'e2e_delay_ms', 'client_rtt_ms', 'server_rtt_ms',
            'client_packet_loss_rate', 'server_packet_loss_rate',
            'page_response_delay', 'page_download_throughput',
            'dns_delay', 'dns_sr', 'tcp_connection_sr', 'number_of_regions',
            # legacy names (pre-v6 KPI dataset fallback)
            'avg_latency_ms', 'avg_packet_loss', 'voip_quality',
            'congestion_level', 'ratio_5g', 'traffic_diversity',
        ] if f in kpi.columns
    ]
    print(f'Statistical features ({len(STAT_FEATURES)}): {STAT_FEATURES}\n')

    # Conservative thresholds for telecom KPI data
    THRESHOLD_Z  = 3.5   # was 2.5 → flagged 64%; 3.5 targets ~10-15%
    IQR_MULT     = 3.0   # was 1.5 → too aggressive on skewed distributions

    stat_flag, stat_driver, stat_score = stat_anomaly_score(
        kpi, STAT_FEATURES,
        threshold_z = THRESHOLD_Z,
        iqr_mult    = IQR_MULT,
    )

    kpi['stat_anomaly'] = stat_flag.values
    kpi['stat_driver']  = stat_driver.values
    kpi['stat_score']   = stat_score.values

    n_stat = stat_flag.sum()
    rate   = stat_flag.mean() * 100
    print(f'Statistical anomalies : {n_stat:,}  ({rate:.1f}%)')
    print(f'  threshold_z={THRESHOLD_Z}, iqr_mult={IQR_MULT}')
    print(f'\nTop drivers (stat):')
    print(stat_driver[stat_flag==1].value_counts().head(8).to_string())
else:
    print('⚠ KPI dataset not loaded')
    kpi = None


Statistical features (11): ['e2e_delay_ms', 'client_rtt_ms', 'server_rtt_ms', 'client_packet_loss_rate', 'server_packet_loss_rate', 'page_response_delay', 'page_download_throughput', 'dns_delay', 'dns_sr', 'tcp_connection_sr', 'number_of_regions']

Statistical anomalies : 3,479  (71.1%)
  threshold_z=3.5, iqr_mult=3.0

Top drivers (stat):
page_response_delay        929
e2e_delay_ms               653
dns_sr                     530
server_rtt_ms              476
client_packet_loss_rate    319
client_rtt_ms              243
tcp_connection_sr          158
server_packet_loss_rate    141


---
## Section 3 — Machine Learning: Isolation Forest

### How Isolation Forest works

```
Core idea: anomalies are EASY TO ISOLATE.

1. Build many random decision trees
2. For each data point, measure how many splits are needed to isolate it
3. Anomalies require FEWER splits (short path length)
4. Normal points require many splits (long path length)

anomaly_score = f(average path length)
Score near 1  → highly anomalous
Score near 0  → normal
```

**Why Isolation Forest for SpiriCam?**
- Works on all 22 KPI features simultaneously (multivariate)
- No assumption about data distribution (unlike Z-score)
- Fast on 4,896 subscribers
- `contamination` parameter controls how many % you expect to be anomalous

In [6]:
# ── 3.1  Isolation Forest on the full KPI feature matrix ──────────────
if kpi is not None:
    # Features for IF — same as churn model minus label cols
    LEAKAGE = {'c1_low_usage', 'c2_low_dur', 'msisdn', 'churn',
               'stat_anomaly', 'stat_driver', 'stat_score'}
    IF_FEATURES = [
        c for c in kpi.select_dtypes(include=[np.number]).columns
        if c not in LEAKAGE
    ]
    print(f'Isolation Forest features ({len(IF_FEATURES)}): {IF_FEATURES[:8]}...\n')

    # Fill NaN with median before fitting
    X_if = kpi[IF_FEATURES].fillna(kpi[IF_FEATURES].median())

    # Scale features (IF works on raw data too, but scaling helps interpretability)
    scaler_if = StandardScaler()
    X_scaled  = scaler_if.fit_transform(X_if)

    # ── Isolation Forest ──────────────────────────────────────────────
    #  contamination: expected % of anomalies
    #  SpiriCam: ~5% seems reasonable for a telecom dataset
    #  Adjust if you see too many / too few alerts on the dashboard.
    iso_forest = IsolationForest(
        n_estimators  = 200,
    contamination = 0.05,   # 5% → 245 anomalies from 4,896 subs ✅
        max_samples   = 'auto',
        random_state  = 42,
        n_jobs        = -1,
    )
    iso_forest.fit(X_scaled)

    # predict: -1 = anomaly, 1 = normal → convert to 0/1
    if_pred  = iso_forest.predict(X_scaled)
    if_flag  = (if_pred == -1).astype(int)

    # Raw anomaly score: lower = more anomalous, negate to make higher = worse
    if_score_raw = iso_forest.score_samples(X_scaled)  # negative anomaly score
    if_score_norm = (-if_score_raw - (-if_score_raw).min()) / \
                   ((-if_score_raw).max() - (-if_score_raw).min() + 1e-8)

    kpi['if_anomaly']   = if_flag
    kpi['if_score']     = if_score_norm

    # Severity based on IF score quintiles among anomalies
    def if_severity(score):
        if score >= 0.85: return 'Critical'
        if score >= 0.65: return 'High'
        if score >= 0.40: return 'Medium'
        return 'Low'

    kpi['if_severity'] = kpi['if_score'].apply(if_severity)
    kpi.loc[kpi['if_anomaly'] == 0, 'if_severity'] = 'Normal'

    n_if = if_flag.sum()
    print(f'Isolation Forest anomalies : {n_if:,}  ({if_flag.mean()*100:.1f}% — target ≈5%)')
    print(f'\nSeverity breakdown:')
    print(kpi[kpi['if_anomaly']==1]['if_severity'].value_counts().to_string())

    # Save model
    joblib.dump(iso_forest,  ANOM_DIR / 'isolation_forest.pkl')
    joblib.dump(scaler_if,   ANOM_DIR / 'if_scaler.pkl')
    joblib.dump(IF_FEATURES, ANOM_DIR / 'if_features.pkl')
    print(f'\n✅ Models saved to {ANOM_DIR}')
else:
    print('⚠ KPI dataset not available — skipping Isolation Forest')

Isolation Forest features (23): ['session_flag', 'traffic_5g', 'dou_total', 'duration', 'dou_total_imputed', 'duration_imputed', 'traffic_5g_imputed', 'traffic_4g_imputed']...

Isolation Forest anomalies : 245  (5.0% — target ≈5%)

Severity breakdown:
if_severity
Medium      145
High         94
Critical      6

✅ Models saved to models\anomaly



Isolation Forest anomalies : 245  (5.0% — target ≈5%)

Severity breakdown:
if_severity
Medium      180
High         60
Critical      5

✅ Models saved to models\anomaly


---
## Section 4 — Consensus Flagging + Top Anomaly Driver

The **consensus flag** is what the NOC dashboard shows.
A row is flagged only when **both** methods agree:

```
anomaly_consensus = 1  iff  stat_anomaly = 1  AND  if_anomaly = 1
```

This dramatically reduces false positives.
The `top_anomaly_driver` field tells the NOC engineer **which feature triggered the alert**.

In [7]:
# ── 4.1  Build consensus + combined score ─────────────────────────────
if kpi is not None:
    # Consensus: both methods must agree
    kpi['anomaly_consensus'] = (
        (kpi['stat_anomaly'] == 1) & (kpi['if_anomaly'] == 1)
    ).astype(int)

    # Combined score: average of normalised stat and IF scores
    kpi['combined_score'] = (
        kpi['stat_score'].fillna(0) * 0.4 +
        kpi['if_score'].fillna(0)   * 0.6
    ).round(4)

    # Final anomaly_flag: either method is enough (union)
    # Use this for broad detection; use consensus for high-precision alerts
    kpi['anomaly_flag'] = (
        (kpi['stat_anomaly'] == 1) | (kpi['if_anomaly'] == 1)
    ).astype(int)

    # Top anomaly driver per subscriber:
    # For each flagged row, find which feature deviates most from its mean
    X_deviation = kpi[IF_FEATURES].fillna(kpi[IF_FEATURES].median())
    X_scaled_df  = pd.DataFrame(
        scaler_if.transform(X_deviation),
        index=kpi.index,
        columns=IF_FEATURES
    )
    kpi['top_anomaly_driver'] = X_scaled_df.abs().idxmax(axis=1)
    # Only keep driver for flagged rows
    kpi.loc[kpi['anomaly_flag'] == 0, 'top_anomaly_driver'] = ''

    # ── Summary ──────────────────────────────────────────────────────
    n_union     = kpi['anomaly_flag'].sum()
    n_consensus = kpi['anomaly_consensus'].sum()
    n_stat_only = ((kpi['stat_anomaly']==1) & (kpi['if_anomaly']==0)).sum()
    n_if_only   = ((kpi['stat_anomaly']==0) & (kpi['if_anomaly']==1)).sum()

    print('══════════════════════════════════════════════════')
    print('ANOMALY DETECTION RESULTS — CONSENSUS ANALYSIS')
    print('══════════════════════════════════════════════════')
    print(f'  Statistical only (Z-score/IQR)  : {kpi["stat_anomaly"].sum():,}')
    print(f'  Isolation Forest only           : {kpi["if_anomaly"].sum():,}')
    print(f'  Union  (either flags)           : {n_union:,}  ({n_union/len(kpi)*100:.1f}%)')
    print(f'  Consensus (both agree) ← ALERTS : {n_consensus:,}  ({n_consensus/len(kpi)*100:.1f}%)')
    print(f'\n  Stat only (not confirmed by IF) : {n_stat_only:,}  ← likely false positives')
    print(f'  IF only  (not confirmed by stat): {n_if_only:,}  ← subtle multivariate anomalies')

    print(f'\nTop anomaly drivers (consensus events):')
    print(kpi[kpi['anomaly_consensus']==1]['top_anomaly_driver'].value_counts().head(8).to_string())

══════════════════════════════════════════════════
ANOMALY DETECTION RESULTS — CONSENSUS ANALYSIS
══════════════════════════════════════════════════
  Statistical only (Z-score/IQR)  : 3,479
  Isolation Forest only           : 245
  Union  (either flags)           : 3,479  (71.1%)
  Consensus (both agree) ← ALERTS : 245  (5.0%)

  Stat only (not confirmed by IF) : 3,234  ← likely false positives
  IF only  (not confirmed by stat): 0  ← subtle multivariate anomalies

Top anomaly drivers (consensus events):
top_anomaly_driver
session_flag               51
traffic_5g_imputed         25
traffic_5g                 25
page_response_delay        24
tcp_connection_sr          21
client_rtt_ms              21
server_packet_loss_rate    14
dns_delay                  13


---
## Section 5 — Use Cases

In [8]:
# ── 5.1  UC-1: Complaint spike detection ──────────────────────────────
# Aggregate complaints by date, then flag days with abnormally high counts

if 'complaints' in datasets:
    complaints = datasets['complaints'].copy()

    # Find date column
    date_cols = [c for c in complaints.columns
                 if 'date' in c.lower() or 'time' in c.lower() or 'day' in c.lower()]
    print(f'Date candidate columns: {date_cols}')

    if date_cols:
        date_col = date_cols[0]
        complaints[date_col] = pd.to_datetime(complaints[date_col], errors='coerce')
        daily = complaints.groupby(complaints[date_col].dt.date).size().reset_index()
        daily.columns = ['date', 'complaint_count']
        daily['date'] = pd.to_datetime(daily['date'])
        daily = daily.sort_values('date')

        # Z-score spike detection on daily counts
        daily['z_score']   = (daily['complaint_count'] - daily['complaint_count'].mean()) / \
                              daily['complaint_count'].std()
        daily['spike_flag']= (daily['z_score'].abs() > 2.0).astype(int)

        # Rolling 7-day baseline comparison
        daily['rolling_7d'] = daily['complaint_count'].rolling(7, min_periods=1).mean()
        daily['vs_baseline_pct'] = (
            (daily['complaint_count'] - daily['rolling_7d']) /
            daily['rolling_7d'].replace(0, 1) * 100
        ).round(1)

        spike_days = daily[daily['spike_flag'] == 1]
        print(f'\nComplaint spike days detected: {len(spike_days)}')
        print(spike_days[['date','complaint_count','z_score','vs_baseline_pct']].to_string(index=False))
    else:
        print('⚠ No date column found in complaints — skipping UC-1')
        daily = None
else:
    print('⚠ Complaints dataset not loaded')
    daily = None

⚠ Complaints dataset not loaded


In [9]:
# ── 5.2  UC-4: Disengagement risk surge ──────────────────────────────
# NB07-FIX-3: v6 schema uses 'risk' (float 0-1) + 'risk_band' (low/medium/high)
# Legacy schema has 'risk_level' (CRITICAL/HIGH/MEDIUM/LOW). Both handled.
if 'scores' in datasets:
    scores = datasets['scores'].copy()
    if 'risk_band' in scores.columns:
        # v6 schema
        risk_dist = scores['risk_band'].value_counts(normalize=True).mul(100).round(2)
        risk_n    = scores['risk_band'].value_counts()
        print('Disengagement risk distribution (v6):')
        for level in ['high', 'medium', 'low']:
            pct = risk_dist.get(level, 0)
            n   = risk_n.get(level, 0)
            bar = '█' * int(pct / 2)
            print(f'  {level.upper():<12s}: {n:5,}  ({pct:.1f}%)  {bar}')
        high_pct   = risk_dist.get('high', 0)
        surge_flag = high_pct > 30
        print(f'\n  HIGH risk = {high_pct:.1f}%')
        if surge_flag:
            print('  ⚠ HIGH DISENGAGEMENT RISK SURGE: HIGH > 30% threshold')
        else:
            print('  ✅ Disengagement risk within normal range')
    elif 'risk_level' in scores.columns:
        # legacy schema
        risk_dist = scores['risk_level'].value_counts(normalize=True).mul(100).round(2)
        risk_n    = scores['risk_level'].value_counts()
        print('Churn risk distribution (legacy):')
        for level in ['CRITICAL','HIGH','MEDIUM','LOW']:
            pct = risk_dist.get(level, 0)
            n   = risk_n.get(level, 0)
            bar = '█' * int(pct / 2)
            print(f'  {level:<12s}: {n:5,}  ({pct:.1f}%)  {bar}')
        high_pct   = risk_dist.get('CRITICAL', 0) + risk_dist.get('HIGH', 0)
        surge_flag = high_pct > 40
        print(f'\n  CRITICAL + HIGH = {high_pct:.1f}%')
        if surge_flag:
            print('  ⚠ CHURN RISK SURGE ALERT: CRITICAL+HIGH > 40% threshold')
        else:
            print('  ✅ Churn risk within normal range')
    else:
        print('  ⚠ Risk column not found (expected risk_band or risk_level)')


# ── 5.3  UC-6: Brand anomaly ──────────────────────────────────────────
# Key fixes vs original:
#   1. Filter brands with < 10 subscribers (single-subscriber brands have extreme
#      averages that are not network anomalies — they are sampling noise)
#   2. IQR multiplier raised to 2.5 (was 1.5) — less sensitive on brand aggregates
if kpi is not None:
    brand_col = next((c for c in kpi.columns if 'brand' in c.lower()), None)
    print(f'\nBrand column: {brand_col}')

    if brand_col:
        brand_map = {}
        for lep in [MODEL_DIR/'le_brand.pkl', MODEL_DIR/'churn_le_brand.pkl']:
            if lep.exists():
                le_b = joblib.load(lep)
                brand_map = {i: n for i, n in enumerate(le_b.classes_)}
                print(f'  Brand encoder: {len(brand_map)} brands')
                break

        metric_cols = [c for c in [
            'ratio_5g','traffic_5g','avg_latency_ms','avg_packet_loss',
            'voip_quality','social_ratio','traffic_diversity',
        ] if c in kpi.columns]

        brand_stats = kpi.groupby(brand_col).agg(
            subscriber_count=(brand_col,'count'),
            **{f'avg_{m}': (m,'mean') for m in metric_cols}
        ).reset_index()

        brand_stats['brand_name'] = brand_stats[brand_col].map(
            lambda x: brand_map.get(int(x), f'Brand_{x}')
        )

        # ── FIX: only analyse brands with ≥ 10 subscribers ───────────
        # Single-subscriber averages are noise, not network anomalies.
        # Raising IQR multiplier to 2.5 further reduces false positives.
        MIN_SUBS = 10
        brand_main = brand_stats[brand_stats['subscriber_count'] >= MIN_SUBS].copy()
        brand_rare = brand_stats[brand_stats['subscriber_count'] <  MIN_SUBS].copy()
        print(f'\n  Brands with ≥{MIN_SUBS} subscribers : {len(brand_main)}')
        print(f'  Brands with  <{MIN_SUBS} (excluded)  : {len(brand_rare)}')

        avg_metric_cols = [f'avg_{m}' for m in metric_cols if f'avg_{m}' in brand_main.columns]
        flag_cols = []
        for col in avg_metric_cols:
            flag_col = f'{col}_anom'
            # IQR multiplier=2.5 — conservative for brand-level aggregates
            brand_main[flag_col] = iqr_anomaly(brand_main[col], multiplier=2.5).astype(int)
            flag_cols.append(flag_col)

        if flag_cols:
            brand_main['brand_anomaly']  = brand_main[flag_cols].any(axis=1).astype(int)
            brand_main['anomaly_driver'] = (
                brand_main[flag_cols].idxmax(axis=1)
                .str.replace('_anom','').str.replace('avg_','')
            )
            brand_main.loc[brand_main['brand_anomaly']==0, 'anomaly_driver'] = ''

            print('\nBrand anomaly analysis (≥10 subscribers, IQR×2.5):')
            show = ['brand_name','subscriber_count'] + avg_metric_cols[:3] + ['brand_anomaly','anomaly_driver']
            show = [c for c in show if c in brand_main.columns]
            print(brand_main[show].sort_values('brand_anomaly',ascending=False)
                  .head(20).round(4).to_string(index=False))

            n_anom = brand_main['brand_anomaly'].sum()
            if n_anom:
                anom_brands = brand_main[brand_main['brand_anomaly']==1]['brand_name'].tolist()
                print(f'\n⚠ BRAND ANOMALIES ({n_anom}): {anom_brands}')
            else:
                print('\n✅ No brand anomalies detected')
    else:
        print('  brand_encoded not found — UC-6 skipped')


Disengagement risk distribution (v6):
  HIGH        :   555  (21.6%)  ██████████
  MEDIUM      :   425  (16.6%)  ████████
  LOW         : 1,586  (61.8%)  ██████████████████████████████

  HIGH risk = 21.6%
  ✅ Disengagement risk within normal range

Brand column: brand
  Brand encoder: 90 brands


ValueError: invalid literal for int() with base 10: 'ACCENT'

In [ ]:
# ── 5.4  UC-5: Geographic hotspot — province anomaly ─────────────────
# FIX: only flag provinces where anomaly_rate is ABOVE the upper IQR fence.
# The original code flagged KEF as a hotspot because it had the LOWEST
# anomaly_rate (0.44 vs mean ~0.64) — IQR flagged the outlier low end.
# For NOC purposes a 'hotspot' is HIGH anomaly, not LOW anomaly.

if kpi is not None:
    province_col = next((c for c in kpi.columns
                        if 'province' in c.lower() or 'region' in c.lower()), None)
    print(f'Province column: {province_col}')

    if province_col:
        province_map_nb = {}
        for lep in [MODEL_DIR/'le_province.pkl', MODEL_DIR/'churn_le_province.pkl']:
            if lep.exists():
                le_p = joblib.load(lep)
                province_map_nb = {i: n for i, n in enumerate(le_p.classes_)}
                print(f'  Province encoder: {len(province_map_nb)} governorates')
                break
        if not province_map_nb:
            _TN = ['Ariana','Béja','Ben Arous','Bizerte','Gabès','Gafsa',
                   'Jendouba','Kairouan','Kasserine','Kébili','Kef','Mahdia',
                   'Manouba','Médenine','Monastir','Nabeul','Sfax','Sidi Bouzid',
                   'Siliana','Sousse','Tataouine','Tozeur','Tunis','Zaghouan']
            province_map_nb = {i: n for i, n in enumerate(_TN)}
            print('  Using 24-governorate alphabetical fallback')

        prov_agg = {}
        if 'anomaly_flag'       in kpi.columns: prov_agg['anomaly_rate']   = ('anomaly_flag',      'mean')
        if 'anomaly_consensus'  in kpi.columns: prov_agg['consensus_rate'] = ('anomaly_consensus', 'mean')
        if 'ratio_5g'           in kpi.columns: prov_agg['avg_ratio_5g']   = ('ratio_5g',          'mean')
        if 'churn'              in kpi.columns: prov_agg['churn_rate']     = ('churn',             'mean')
        prov_agg['subscribers'] = (province_col, 'count')

        province_stats = kpi.groupby(province_col).agg(**prov_agg).reset_index()
        province_stats['province'] = province_stats[province_col].apply(
            lambda x: province_map_nb.get(int(x), f'Province {x}')
        )

        if 'anomaly_rate' in province_stats.columns:
            ar = province_stats['anomaly_rate']
            # ── FIX: compute upper fence only ─────────────────────────
            q1, q3 = ar.quantile(0.25), ar.quantile(0.75)
            upper_fence = q3 + 1.5 * (q3 - q1)
            province_stats['hotspot'] = (ar > upper_fence).astype(int)

            province_stats = province_stats.sort_values('anomaly_rate', ascending=False)
            show = ['province','subscribers','anomaly_rate','consensus_rate',
                    'avg_ratio_5g','churn_rate','hotspot']
            show = [c for c in show if c in province_stats.columns]

            print(f'\n  IQR upper fence = {upper_fence:.4f}')
            print('  Provinces above upper fence = geographic hotspots (HIGH anomaly rate)')
            print()
            print(province_stats[show].round(4).to_string(index=False))

            hotspots = province_stats[province_stats['hotspot']==1]
            low_anomaly = province_stats[province_stats['anomaly_rate'] < ar.quantile(0.25)]
            if len(hotspots):
                print(f'\n⚠ GEOGRAPHIC HOTSPOTS ({len(hotspots)}): {list(hotspots["province"])}')
                print('   These provinces have anomaly_rate > upper IQR fence — require attention')
            else:
                print('\n✅ No province hotspots — anomaly_rate is uniformly distributed')
                print(f'   Range: {ar.min():.3f} – {ar.max():.3f}  (tight spread = no outlier regions)')
            if len(low_anomaly):
                print(f'\n  Low-anomaly provinces (below Q1={ar.quantile(0.25):.3f}):'
                      f' {list(low_anomaly["province"])} ← best-performing regions')
        else:
            print('  anomaly_flag not yet computed — run Section 4 first')
    else:
        print('  province_encoded / region not found in KPI dataset')
        province_stats = None
else:
    print('⚠ KPI dataset not available')
    province_stats = None


---
## Section 6 — Visualisations

In [ ]:
# ── 6.1  Combined score distribution — normal vs anomaly ──────────────
if kpi is not None and 'combined_score' in kpi.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Anomaly Detection Overview — SpiriCam KPI Dataset',
                 fontsize=14, fontweight='bold', color=HW['navy'])

    # A: Score distribution
    ax = axes[0]
    normal   = kpi[kpi['anomaly_flag']==0]['combined_score']
    flagged  = kpi[kpi['anomaly_flag']==1]['combined_score']
    ax.hist(normal,  bins=40, color=HW['blue'],  alpha=0.7, label=f'Normal ({len(normal):,})')
    ax.hist(flagged, bins=40, color=HW['red'],   alpha=0.8, label=f'Anomaly ({len(flagged):,})')
    ax.set_title('Combined Anomaly Score Distribution')
    ax.set_xlabel('Combined Score'); ax.set_ylabel('Count')
    ax.legend()

    # B: Severity breakdown
    ax = axes[1]
    sev_order  = ['Critical','High','Medium','Low','Normal']
    sev_colors = [HW['red'], HW['amber'], HW['blue'], HW['green'], '#9CA3AF']
    sev_counts = kpi['if_severity'].value_counts().reindex(sev_order, fill_value=0)
    bars = ax.bar(sev_counts.index, sev_counts.values,
                  color=sev_colors[:len(sev_counts)], alpha=0.88)
    for bar, val in zip(bars, sev_counts.values):
        if val > 0:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
                    str(val), ha='center', fontsize=9, fontweight='bold')
    ax.set_title('Anomaly Severity Distribution (Isolation Forest)')
    ax.set_ylabel('Count')

    # C: Top 8 anomaly drivers
    ax = axes[2]
    drivers = kpi[kpi['anomaly_flag']==1]['top_anomaly_driver'].value_counts().head(8)
    if len(drivers):
        colors_d = [HW['red'] if i==0 else HW['blue'] for i in range(len(drivers))]
        ax.barh(drivers.index[::-1], drivers.values[::-1], color=colors_d[::-1], alpha=0.88)
        ax.set_title('Top Anomaly Drivers')
        ax.set_xlabel('Count')
        ax.grid(axis='x', alpha=0.4); ax.grid(axis='y', visible=False)

    watermark(fig)
    plt.tight_layout()
    save_fig('fig07_A_anomaly_overview')
    plt.show()

In [ ]:
# ── 6.2  Scatter: two key features coloured by anomaly status ─────────
if kpi is not None:
    feat_pairs = [
        ('ratio_5g',     'avg_latency_ms'),
        ('ratio_5g',     'avg_packet_loss'),
        ('traffic_5g',   'avg_latency_ms'),
    ]
    available_pairs = [(a, b) for a, b in feat_pairs
                       if a in kpi.columns and b in kpi.columns]

    if available_pairs:
        n_plots = min(len(available_pairs), 3)
        fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 5))
        if n_plots == 1: axes = [axes]
        fig.suptitle('Anomaly Scatter — Feature Pairs Coloured by Anomaly Status',
                     fontsize=13, fontweight='bold', color=HW['navy'])

        for ax, (fa, fb) in zip(axes, available_pairs[:n_plots]):
            normal  = kpi[kpi['anomaly_flag']==0]
            anomaly = kpi[kpi['anomaly_flag']==1]
            ax.scatter(normal[fa],  normal[fb],  c=HW['blue'], alpha=0.3, s=5,  label='Normal')
            ax.scatter(anomaly[fa], anomaly[fb], c=HW['red'],  alpha=0.7, s=12, label='Anomaly', zorder=3)
            ax.set_xlabel(fa.replace('_',' '))
            ax.set_ylabel(fb.replace('_',' '))
            ax.set_title(f'{fa} vs {fb}')
            ax.legend(markerscale=2, fontsize=9)

        watermark(fig)
        plt.tight_layout()
        save_fig('fig07_B_anomaly_scatter')
        plt.show()
    else:
        print('⚠ Feature pairs not available for scatter plot')

In [ ]:
# ── 6.3  Complaint spike timeline ─────────────────────────────────────
if daily is not None and len(daily) > 0:
    fig, ax = plt.subplots(figsize=(14, 5))

    ax.plot(daily['date'], daily['complaint_count'],
            color=HW['blue'], linewidth=1.5, zorder=2, label='Daily complaints')
    ax.plot(daily['date'], daily['rolling_7d'],
            color=HW['muted'], linewidth=1, linestyle='--', label='7-day rolling avg')

    # Shade spike days
    spikes = daily[daily['spike_flag']==1]
    ax.scatter(spikes['date'], spikes['complaint_count'],
               color=HW['red'], s=60, zorder=3, label=f'Spike alert ({len(spikes)})', marker='^')

    # +/- 2 std bands
    mu  = daily['complaint_count'].mean()
    std = daily['complaint_count'].std()
    ax.axhline(mu + 2*std, color=HW['amber'], linewidth=0.8, linestyle=':', alpha=0.7, label='+2σ threshold')
    ax.axhline(mu,         color=HW['green'], linewidth=0.8, linestyle=':', alpha=0.5, label='Mean')

    ax.set_title('Complaint Spike Detection — Z-Score Method', pad=12)
    ax.set_xlabel('Date')
    ax.set_ylabel('Daily Complaint Count')
    ax.legend(fontsize=9)
    plt.xticks(rotation=30, ha='right')

    watermark(fig)
    plt.tight_layout()
    save_fig('fig07_C_complaint_spikes')
    plt.show()
else:
    print('⚠ Daily complaint data not available for timeline')

In [ ]:
# ── 6.4  Province heatmap — anomaly rate by governorate ───────────────
if province_stats is not None and 'province' in province_stats.columns:
    prov = province_stats.sort_values('anomaly_rate', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = [HW['red'] if h else HW['blue']
              for h in prov['hotspot'] if 'hotspot' in prov.columns]

    bars = ax.barh(prov['province'], prov['anomaly_rate'] * 100,
                   color=colors if colors else HW['blue'], alpha=0.85)

    for bar, val in zip(bars, prov['anomaly_rate'] * 100):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=9)

    ax.set_title('Anomaly Rate by Governorate — Geographic Hotspot Analysis', pad=12)
    ax.set_xlabel('Anomaly Rate (%)')
    ax.grid(axis='x', alpha=0.4); ax.grid(axis='y', visible=False)

    if 'hotspot' in prov.columns:
        red_patch   = mpatches.Patch(color=HW['red'],  label='Hotspot (IQR outlier)')
        blue_patch  = mpatches.Patch(color=HW['blue'], label='Normal range')
        ax.legend(handles=[red_patch, blue_patch], fontsize=9)

    watermark(fig)
    plt.tight_layout()
    save_fig('fig07_D_province_hotspots')
    plt.show()
else:
    print('⚠ Province stats not available')

---
## Section 7 — Build `anomaly_results.parquet`

This is the file the API reads.
It must contain these exact columns to feed `AnomalyFeed.jsx`:

| Column | Type | Description |
|--------|------|-------------|
| `anomaly_flag` | 0/1 | Union flag (either method) |
| `anomaly_consensus` | 0/1 | Both methods agree |
| `if_anomaly` | 0/1 | Isolation Forest flag |
| `stat_anomaly` | 0/1 | Statistical flag |
| `combined_score` | float | Normalised 0–1 urgency |
| `if_severity` | str | Normal / Low / Medium / High / Critical |
| `top_anomaly_driver` | str | Feature name that triggered the alert |
| `region` | str | Governorate name (for regions chart) |
| `date` | datetime | Synthetic date (used for timeline) |

In [ ]:
# ── 7.1  Assemble the final output dataframe ───────────────────────────
if kpi is not None:
    # Required output columns.
    # NB07-FIX-4: msisdn added — without it the NOC cannot identify which
    # subscriber is anomalous; the dashboard and notification scanner would
    # receive aggregate flags with no customer identity attached.
    _has_msisdn = 'msisdn' in kpi.columns
    REQUIRED = (
        (['msisdn'] if _has_msisdn else []) +
        ['anomaly_flag', 'anomaly_consensus', 'if_anomaly', 'stat_anomaly',
         'combined_score', 'if_severity', 'top_anomaly_driver']
    )
    if not _has_msisdn:
        print('⚠ msisdn column not found in kpi — subscriber identity missing')

    results = kpi[REQUIRED].copy()

    # ── Add region (province name) ────────────────────────────────────
    if 'province_encoded' in kpi.columns:
        province_map_local = {}
        for lep in [MODEL_DIR/'le_province.pkl', MODEL_DIR/'churn_le_province.pkl']:
            if lep.exists():
                le_p = joblib.load(lep)
                province_map_local = {i: n for i, n in enumerate(le_p.classes_)}
                break
        if not province_map_local:
            _TN = ['Ariana','Béja','Ben Arous','Bizerte','Gabès','Gafsa',
                   'Jendouba','Kairouan','Kasserine','Kébili','Kef','Mahdia',
                   'Manouba','Médenine','Monastir','Nabeul','Sfax','Sidi Bouzid',
                   'Siliana','Sousse','Tataouine','Tozeur','Tunis','Zaghouan']
            province_map_local = {i: n for i, n in enumerate(_TN)}
        results['region'] = kpi['province_encoded'].map(
            lambda x: province_map_local.get(int(x), f'Province {x}')
        )
    else:
        results['region'] = 'Unknown'

    # ── Add date (synthetic: distribute across recent 90 days) ────────
    # KPI dataset is a snapshot, not time-series.
    # We assign dates to simulate a timeline for the dashboard.
    # Anomalous events are clustered near recent dates.
    base_date = datetime.today()
    np.random.seed(42)

    n = len(results)
    normal_dates  = [base_date - timedelta(days=int(d))
                     for d in np.random.uniform(0, 90, n)]

    # Anomalies skewed toward more recent dates (last 30 days)
    anom_idx = results[results['anomaly_flag']==1].index
    for idx in anom_idx:
        pos = results.index.get_loc(idx)
        normal_dates[pos] = base_date - timedelta(
            days=int(np.random.uniform(0, 30))
        )

    results['date'] = normal_dates

    # ── Final dtypes ──────────────────────────────────────────────────
    results['anomaly_flag']       = results['anomaly_flag'].astype(int)
    results['anomaly_consensus']  = results['anomaly_consensus'].astype(int)
    results['if_anomaly']         = results['if_anomaly'].astype(int)
    results['stat_anomaly']       = results['stat_anomaly'].astype(int)
    results['combined_score']     = results['combined_score'].astype(float).round(4)
    results['if_severity']        = results['if_severity'].astype(str)
    results['top_anomaly_driver'] = results['top_anomaly_driver'].astype(str)
    results['region']             = results['region'].astype(str)
    results['date']               = pd.to_datetime(results['date'])

    # ── Save ─────────────────────────────────────────────────────────
    out_path = ANOM_DIR / 'anomaly_results.parquet'
    results.to_parquet(out_path, index=False)

    # NB07-FIX-4b: also save to data/outputs/ so notifications_api.py
    # scanner (which reads anomaly_results.parquet from that directory)
    # and analytics_api.py find the updated results without code changes.
    out_path_api = OUT_DIR / 'anomaly_results.parquet'
    results.to_parquet(out_path_api, index=False)

    print('Final anomaly_results.parquet:')
    print(f'  Rows           : {len(results):,}')
    print(f'  Columns        : {list(results.columns)}')
    print(f'  -> {out_path}')
    print(f'  -> {out_path_api}  (API compatibility)')
    print()
    print(results[results['anomaly_consensus']==1][
        ['region','if_severity','combined_score','top_anomaly_driver','date']
    ].sort_values('combined_score', ascending=False).head(10).to_string(index=False))
else:
    print('❌ KPI dataset missing — cannot build anomaly_results.parquet')
    print('   Ensure churn_features.parquet is present and re-run this notebook')

In [ ]:
# ── 7.2  Final summary ────────────────────────────────────────────────
if kpi is not None:
    figs = sorted(FIG_DIR.glob('fig07_*.png'))

    # Re-read actual values from the kpi dataframe (reflects current run)
    n_total      = len(kpi)
    n_stat       = int(kpi['stat_anomaly'].sum())       if 'stat_anomaly'      in kpi.columns else 0
    n_if         = int(kpi['if_anomaly'].sum())         if 'if_anomaly'        in kpi.columns else 0
    n_consensus  = int(kpi['anomaly_consensus'].sum())  if 'anomaly_consensus' in kpi.columns else 0
    rate_stat    = n_stat / n_total * 100
    rate_if      = n_if   / n_total * 100
    rate_cons    = n_consensus / n_total * 100

    print('\n' + '═'*65)
    print('NOTEBOOK 07 — ANOMALY DETECTION COMPLETE')
    print('═'*65)
    print()
    print('  Methods applied:')
    print('    Statistical (Z-Score + IQR)    ← threshold_z=3.5  iqr_mult=3.0')
    print('    Isolation Forest               ← contamination=5%  n_estimators=200')
    print('    Consensus (both agree)         ← high-precision NOC alerts')
    print()
    print(f'  Total subscribers analysed  : {n_total:,}')
    print(f'  Statistical anomalies       : {n_stat:,}  ({rate_stat:.1f}%)')
    if rate_stat > 20:
        print(f'    ⚠ Rate still high — consider raising threshold_z above 3.5')
        print(f'      or using log-transform on latency before detection')
    else:
        print(f'    ✅ Rate within expected 5-15% range')
    print(f'  Isolation Forest anomalies  : {n_if:,}  ({rate_if:.1f}%)')
    print(f'  Consensus alerts (both)     : {n_consensus:,}  ({rate_cons:.1f}%)')

    # Quality check: consensus should be meaningful (not just = IF)
    if n_consensus == n_if:
        print(f'    ⚠ Consensus = IF exactly → stat threshold still too loose')
        print(f'      Raise threshold_z to 4.0 or 5.0 to tighten stat detection')
    elif n_consensus > 0:
        overlap_pct = n_consensus / n_if * 100
        print(f'    ✅ Consensus covers {overlap_pct:.1f}% of IF alerts — two-method agreement working')

    print()
    print('  Severity (Isolation Forest):')
    for sev in ['Critical','High','Medium','Low']:
        n = int((kpi['if_severity']==sev).sum()) if 'if_severity' in kpi.columns else 0
        bar = '█' * int(n / max(n_if, 1) * 20)
        print(f'    {sev:<12s}: {n:3,}  {bar}')
    print()
    print('  Top consensus anomaly drivers:')
    if 'top_anomaly_driver' in kpi.columns and 'anomaly_consensus' in kpi.columns:
        drivers = (kpi[kpi['anomaly_consensus']==1]['top_anomaly_driver']
                   .value_counts().head(6))
        for feat, count in drivers.items():
            bar = '█' * int(count / drivers.max() * 20)
            print(f'    {feat:<30s}: {count:3,}  {bar}')
    print()
    print('  Artifacts saved:')
    print(f'    {ANOM_DIR}/anomaly_results.parquet   ← feeds /api/analytics/anomalies/*')
    print(f'    {ANOM_DIR}/isolation_forest.pkl      ← reusable for inference')
    print(f'    {ANOM_DIR}/if_scaler.pkl')
    print(f'    {ANOM_DIR}/if_features.pkl')
    print()
    print(f'  Figures ({len(figs)}):')
    for f in figs:
        kb = f.stat().st_size // 1024
        print(f'    {f.name:<48s}  {kb:>4} KB')
    print()
    print('  Dashboard endpoints ready:')
    print('    GET /api/analytics/anomalies/summary')
    print('    GET /api/analytics/anomalies/timeline')
    print('    GET /api/analytics/anomalies/regions')
    print()
    print('  ✅ SpiriCom PFE 2026 — all 7 notebooks complete')


---
## Summary

### What was detected

| Detection Layer | Method | Use case |
|----------------|--------|----------|
| Per-feature outliers | Z-Score + IQR | Latency spikes, packet loss surges |
| Multivariate patterns | Isolation Forest | Unusual subscriber behaviour profiles |
| High-confidence alerts | Consensus (both agree) | NOC dashboard — CRITICAL / HIGH severity |
| Complaint spikes | Z-Score on daily counts | UC-1 complaint surge detection |
| Geographic hotspots | IQR on province anomaly rates | UC-5 governorate-level surges |
| Brand anomalies | IQR on brand metrics | UC-6 device brand degradation |

### Dashboard integration

```
NB07 outputs
  └── models/anomaly/anomaly_results.parquet
          ↓
      analytics_api.py  (already reads this file)
          ↓
      GET /api/analytics/anomalies/summary   → KPI tiles
      GET /api/analytics/anomalies/timeline  → Line chart by severity
      GET /api/analytics/anomalies/regions   → Province bar chart
          ↓
      AnomalyFeed.jsx  (already built — will auto-populate)
```

### Real-time vs Batch

| Mode | Recommendation | SpiriCam approach |
|------|---------------|-------------------|
| **Batch** | Re-run NB07 daily/weekly | ✅ Appropriate for KPI snapshot data |
| **Online** | Stream new KPIs through `isolation_forest.pkl` | Future extension — load model + scaler, call `predict()` on new rows |

---
*SpiriCam · NOC Intelligence Platform · Huawei Technologies Tunisia · PFE 2026*